# QuantumClassify Lab — QFF Phase 1 (PS-Q1)

This notebook is the end-to-end reproducible architecture benchmark required by the Qiskit Fall Fest 2026 Phase 1 Master Guide.

**Formal problem statement:** PS-Q1, Variational Quantum Classifier (VQC). The existing Streamlit app and the three model choices are retained. In this official architecture experiment, the classical RBF-SVM is a CPU baseline, and the same VQC type is measured on ideal simulation, Processor A (5 qubits) and Processor B (9 qubits).

The guide's default processor baseline is optimization level 1, transpiler seed 42 and 4096 shots. Full VQC optimisation can be computationally expensive; only use reduced shots or fewer iterations for a clearly labelled smoke test, then restore the official shot count for submission.

No benchmark scores are embedded as example outputs. The next cells compute them from actual runs.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

from src.problem import make_ps_q1_data
from src.processors import load_processor
from src.visualization import plot_coupling_graphs
from src.architecture_benchmark import run_ps_q1_benchmark

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

SEED = 42
SHOTS = 4096
MAXITER = 20
FEATURE_MAP = "ZZ entangling"
ANSATZ = "Real Amplitudes"

print("Repository root:", ROOT)
print("PS-Q1 configuration:", {"seed": SEED, "shots": SHOTS, "maxiter": MAXITER, "feature_map": FEATURE_MAP, "ansatz": ANSATZ})


## 1. Generate the official dataset

The 200-sample dataset is generated locally with four features, three informative features and a binary target. An explicit `n_redundant=0` is necessary because scikit-learn otherwise defaults to two redundant features, which would conflict with the four-feature constraint. The split is stratified 80/20. Standardization and angle scaling are fitted on the training split only.


In [ ]:
data = make_ps_q1_data(seed=SEED)
print("Train features:", data.X_train.shape)
print("Test features:", data.X_test.shape)
print("Train labels:", data.y_train.shape)
print("Test labels:", data.y_test.shape)
print("Unique train classes:", sorted(set(data.y_train.tolist())))
print("Quantum feature angle range:", float(data.X_train_q.min()), float(data.X_train_q.max()))


## 2. Inspect the participant-designed processor topologies

Both models use the required `rz`, `sx`, `x`, `cx` basis, optimization level 1, transpiler seed 42, 4096 shots, and identical synthetic noise rates: one-qubit depolarizing 0.001, CX depolarizing 0.005, and readout error 0.01. The selected graph structures are a design choice and are documented in the JSON files.


In [ ]:
processor_A = load_processor("A")
processor_B = load_processor("B")
display(pd.DataFrame([
    {"Processor": p["name"], "Physical qubits": p["num_qubits"],
     "Undirected edges": len(p["coupling_edges"]), "Basis": ", ".join(p["basis_gates"]),
     "Optimization level": p["transpilation"]["optimization_level"],
     "Transpiler seed": p["transpilation"]["seed_transpiler"],
     "Shots": p["shots"], "1q p": p["noise"]["one_qubit_depolarizing_probability"],
     "CX p": p["noise"]["cx_depolarizing_probability"],
     "Readout p": p["noise"]["readout_error_probability"],
     "Design rationale": p["design_rationale"]}
    for p in (processor_A, processor_B)
]))
fig = plot_coupling_graphs([processor_A, processor_B])
display(fig)


## 3. Run the benchmark

This trains the classical baseline once and VQC on the ideal simulator, Processor A and Processor B. The processor runs use the same data split, VQC family, feature map, ansatz, seed, iteration budget and official synthetic noise rates. It also compiles a fixed representative feature-map-plus-ansatz circuit and measures its logical/compiled depth, CX counts, routing SWAPs and ideal/noisy output-distribution fidelity.


In [ ]:
benchmark = run_ps_q1_benchmark(
    shots=SHOTS,
    maxiter=MAXITER,
    seed=SEED,
    feature_map_name=FEATURE_MAP,
    ansatz_name=ANSATZ,
)


## 4. Results — measured by this run

The tables below are populated from the experiment. If a processor-specific model run fails, its row remains marked failed rather than being filled with a made-up accuracy.


In [ ]:
display(Markdown("### Model comparison"))
display(benchmark["models"])
display(Markdown("### Processor A/B architecture comparison"))
display(benchmark["architecture"])

print("Generated artifacts:")
for path in [
    ROOT / "results/tables/model_comparison.csv",
    ROOT / "results/tables/processor_architecture_comparison.csv",
    ROOT / "results/tables/benchmark_summary.json",
    ROOT / "results/figures/processor_coupling_graphs.png",
    ROOT / "results/figures/processor_metric_comparison.png",
    ROOT / "report.md",
]:
    print(path, "exists =", path.exists())


## 5. Interpret carefully

- The CPU RBF-SVM is an algorithmic baseline, not a circuit mapped to Processor A or B.
- Higher depth or CX count indicates greater compiled gate cost for this circuit, not automatically lower classification accuracy.
- Routing SWAP count and additional CX count are reported separately because native-basis translation can decompose SWAPs into CX gates.
- Output-distribution fidelity is the squared Bhattacharyya coefficient between ideal/noisy measurement distributions of one fixed representative circuit. It is not quantum-state fidelity or hardware-measured fidelity.
- Simulator execution time is not a real-device timing result. These experiments alone do not prove quantum advantage.
- The existing interactive dashboard still has all three model types. The quantum-kernel SVM remains a supplementary dashboard experiment and is not substituted for the official PS-Q1 task.


In [ ]:
display(Markdown(benchmark["report"]))
